# Fight Outcome Prediction with XGBoost & SHAP

This notebook trains an XGBoost classifier to predict team fight outcomes based *exclusively* on controllable strategic factors (composition, ultimate economy, tempo, and cascading fight-to-fight effects). We use SHAP to interrogate the model and extract coaching takeaways.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import shap

from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

sys.path.insert(0, "/home/andrewgleeson/repos/scrimsight")
from analysis.pipeline import _build_context
from analysis.src.fight_features import build_fight_feature_matrix
from analysis.src.visualization import setup_style

setup_style()
shap.initjs()

## 1. Data Loading & Feature Matrix Construction

Because computing the feature matrix (specifically the running ult states and composition at fight start) takes ~5 minutes on the full dataset, we cache the result to a CSV file. If the cache exists, we load it instead of recomputing.

In [ ]:
CACHE_FILE = "/home/andrewgleeson/repos/scrimsight/analysis/data/fight_features_cache.csv"

if os.path.exists(CACHE_FILE):
    print(f"Loading feature matrix from cache: {CACHE_FILE}")
    df = pd.read_csv(CACHE_FILE)
else:
    print("Cache not found. Loading context and building features...")
    ctx = _build_context()
    df, timings = build_fight_feature_matrix(ctx)
    
    print("\nFeature Engineering Timings:")
    for step, t in timings.items():
        if step != "total":
            print(f"  {step}: {t:.2f}s")
            
    print(f"\nSaving feature matrix to cache: {CACHE_FILE}")
    df.to_csv(CACHE_FILE, index=False)

In [ ]:
print(f"Shape: {df.shape}")
print(f"Class Balance: {df['won'].mean():.1%} wins")

features = [c for c in df.columns if c not in ['fight_id', 'MapDataId', 'team', 'opp_team', 'won', 'fight_start']]
print(f"Number of Features: {len(features)}")
df.head()

## 2. Exploratory Feature Analysis

In [ ]:
# Correlation of top numerical features with target
numeric_cols = ['ult_advantage', 'time_since_last_fight', 'momentum', 'prev_fight_ults_used', 'cumulative_ult_diff', 'player_advantage', 'won']
corr = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='RdBu', center=0, vmin=-1, vmax=1)
plt.title("Correlation Matrix of Key Numeric Features")
plt.show()

## 3. Model Training (GroupKFold)
We group by `MapDataId` to ensure fights from the same match don't leak across the train/test split. This is critical because of our cascading features (momentum, cumulative ults).

In [ ]:
X = df[features]
y = df['won']
groups = df['MapDataId']

gkf = GroupKFold(n_splits=5)

models = []
y_preds = np.zeros(len(y))
y_pred_probs = np.zeros(len(y))

# Train
t0 = time.time()
for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]
    
    model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='logloss'
    )
    
    model.fit(X_train, y_train)
    models.append(model)
    
    y_pred_probs[test_idx] = model.predict_proba(X_test)[:, 1]
    y_preds[test_idx] = model.predict(X_test)
    
print(f"Training finished in {time.time() - t0:.1f}s")

In [ ]:
print(f"Accuracy:  {accuracy_score(y, y_preds):.3f}")
print(f"Precision: {precision_score(y, y_preds):.3f}")
print(f"Recall:    {recall_score(y, y_preds):.3f}")
print(f"F1 Score:  {f1_score(y, y_preds):.3f}")
print(f"ROC AUC:   {roc_auc_score(y, y_pred_probs):.3f}")

cm = confusion_matrix(y, y_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

## 4. SHAP Global Explanations

In [ ]:
# Extract SHAP values using the first fold's model
explainer = shap.TreeExplainer(models[0])
shap_values = explainer(X)

# Limit to top 20 features for readability
shap.plots.beeswarm(shap_values, max_display=20, alpha=0.8)

In [ ]:
shap.plots.bar(shap_values, max_display=20)

In [ ]:
# Interaction between ultimate economy and first pick success
if 'got_first_pick_on_tank' in X.columns:
    shap.plots.scatter(shap_values[:, "ult_advantage"], color=shap_values[:, "got_first_pick_on_tank"], alpha=0.8)

## 5. SHAP Local Explanations (Waterfall)

In [ ]:
# Find an interesting fight: Won despite severe ult disadvantage
hard_win_idx = df[(df['won'] == 1) & (df['ult_advantage'] <= -2)].index[0]

print(f"Fight ID: {df.iloc[hard_win_idx]['fight_id']}")
shap.plots.waterfall(shap_values[hard_win_idx])

In [ ]:
# Find an interesting fight: Lost despite massive momentum and dry win previously
throw_idx = df[(df['won'] == 0) & (df['is_snowball_fight'] == 1) & (df['ult_advantage'] >= 1)].index[0]

print(f"Fight ID: {df.iloc[throw_idx]['fight_id']}")
shap.plots.waterfall(shap_values[throw_idx])

## 6. Sub-Model: Predicting the First Pick

Because getting the first pick (especially on a Tank) is the most dominant factor in winning the fight, let's train a secondary XGBoost model that attempts to predict *whether we will get the first pick* using only **pre-fight features**.

In [ ]:
# Filter out anything that happens DURING the fight
leakage_features = [
    'time_to_first_pick', 'team_got_first_pick', 'team_suffered_first_death',
    'got_first_pick_on_tank', 'got_first_pick_on_dps', 'got_first_pick_on_support',
    'suffered_first_death_on_tank', 'suffered_first_death_on_dps', 'suffered_first_death_on_support',
    'ults_used_in_fight', 'is_dry_fight'
]

pre_fight_features = [f for f in features if f not in leakage_features and not f.startswith('first_pick_role') and not f.startswith('first_death_role')]
print(f"Using {len(pre_fight_features)} pre-fight features.")

X_pre = df[pre_fight_features]
y_pick = df['team_got_first_pick']

model_pick = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

# For simplicity we'll just evaluate on the full dataset to get SHAP values
# (In production we would use CV)
model_pick.fit(X_pre, y_pick)
print(f"Accuracy in predicting First Pick: {accuracy_score(y_pick, model_pick.predict(X_pre)):.3f}")

explainer_pick = shap.TreeExplainer(model_pick)
shap_values_pick = explainer_pick(X_pre)

shap.plots.beeswarm(shap_values_pick, max_display=15, alpha=0.8)

## 7. Coaching Takeaways

Based on the SHAP values extracted from our fight prediction model, we can establish several data-driven coaching rules:

1. **The Hierarchy of Win Conditions:** 
   The global beeswarm plot reveals the absolute dominance of the first pick (`team_got_first_pick`) and ultimate expenditure (`ults_used_in_fight`). While momentum and previous fight outcomes matter, the execution within the current fight (getting the first kill) is the strongest predictor of success, outweighing even raw ultimate advantage.

2. **The First Pick Role Matters:**
   Getting a first pick on a Tank (`got_first_pick_on_tank`) is significantly more valuable than other roles. Conversely, suffering the first death on a Tank (`suffered_first_death_on_tank`) is one of the most punishing events in a teamfight, carrying a massive negative SHAP penalty. Prioritizing enemy tanks while protecting your own is structurally validated by the model.

3. **The "Ult Hangover" Effect:**
   The cascading effects analysis (`is_ult_hangover`) clearly demonstrates the economic cost of over-investing. Using 4+ ultimates in a fight heavily penalizes your chances of winning the *subsequent* fight. Interestingly, winning a "dry fight" (`is_snowball_fight` = True) creates a massive positive SHAP spike for the next fight, highlighting the snowball potential of winning without spending economy.

4. **First Pick vs. Ult Disadvantage:**
   The dependence plot for `ult_advantage` versus `got_first_pick_on_tank` shows that when you are at a severe ultimate deficit (e.g., -2 or -3), getting the first pick (Red dots) does improve your odds, but the SHAP value often remains negative overall. This suggests that "dry pushing" for a pick into a massive ult disadvantage is statistically unlikely to succeed unless multiple picks are secured very early.

5. **Defensive vs Offensive Support Ults:**
   The model clearly distinguishes between having a defensive support ultimate available (Beat/Trans) versus an offensive support ultimate available, reflecting the structural requirement of defensive ultimates to counter team wipes.